In [2]:
# ========= SARAP PROTOTIPO COMPLETO =========

# ---------- DATA ----------
MUSIC_LIBRARY = [
    "Aver",
    "Dnd",
    "Tú eres mi canción",
    "No te lloro tanto",
]

PAGES = {
    "home": ["Music", "Buscador"],
    "music_main": ["Music", "Selecciona o busca canción"],
    "music_playing": [],  # dinámico
    "music_search": [],   # dinámico
    "search": [],         # dinámico
}

# ---------- STATE ----------
state = "home"
page = 1
slots = {
    "query": None,
    "results": [],
    "playing": None,
}

# ---------- RENDER ----------
def render():
    print(f"\n--- STATE: {state} ---")
    if state == "home":
        for l in PAGES["home"]: print(l)

    elif state == "music_main":
        print("Biblioteca:")
        for m in MUSIC_LIBRARY: print("-", m)

    elif state == "music_playing":
        print("Reproduciendo:")
        print(slots["playing"])

    elif state == "music_search":
        print("Buscador música:")
        print("Query:", slots["query"])
        print("Resultados:")
        for r in slots["results"]: print("-", r)

    elif state == "search":
        print("Buscador general:")
        print("Query:", slots["query"])

    print("-----------------------\n")

# ---------- ACTIONS ----------
def step(action, value=None):
    global state, slots

    # HOME
    if state == "home":
        if action == "music": state = "music_main"
        elif action == "search": state = "search"
        else: return False

    # MUSIC MAIN
    elif state == "music_main":
        if action == "play":
            slots["playing"] = value
            state = "music_playing"
        elif action == "search_music":
            slots["query"] = value
            slots["results"] = [m for m in MUSIC_LIBRARY if value.lower() in m.lower()]
            state = "music_search"
        elif action == "back": state = "home"
        else: return False

    # MUSIC SEARCH
    elif state == "music_search":
        if action == "play":
            slots["playing"] = value
            state = "music_playing"
        elif action == "modify":
            slots["query"] = value
            slots["results"] = [m for m in MUSIC_LIBRARY if value.lower() in m.lower()]
        elif action == "back": state = "music_main"
        else: return False

    # MUSIC PLAYING
    elif state == "music_playing":
        if action == "back": state = "music_main"
        else: return False

    # SEARCH
    elif state == "search":
        if action == "write":
            slots["query"] = value
        elif action == "modify":
            slots["query"] = value
        elif action == "delete":
            slots["query"] = None
        elif action == "back": state = "home"
        else: return False

    return True

# ---------- ROUTER (LLM SIMULADO) ----------
def decide(user):
    u = user.lower()

    if state == "home":
        if "music" in u or "música" in u: return ("music", None)
        if "busc" in u: return ("search", None)

    if state == "music_main":
        if "busca" in u:
            return ("search_music", u.split("busca")[-1].strip())
        for m in MUSIC_LIBRARY:
            if m.lower() in u:
                return ("play", m)
        if "atrás" in u: return ("back", None)

    if state == "music_search":
        for m in slots["results"]:
            if m.lower() in u:
                return ("play", m)
        if "modifica" in u:
            return ("modify", u.split()[-1])
        if "atrás" in u: return ("back", None)

    if state == "music_playing":
        if "atrás" in u: return ("back", None)

    if state == "search":
        if "escribe" in u or "busca" in u:
            return ("write", u.split()[-1])
        if "modifica" in u:
            return ("modify", u.split()[-1])
        if "borra" in u:
            return ("delete", None)
        if "atrás" in u: return ("back", None)

    return (None, None)

# ---------- LOOP ----------
render()
while True:
    try:
        user = input("USER> ")
        if user.lower() in ["exit", "salir"]: break

        action, value = decide(user)
        if not action:
            print("ACTION: none => FAIL")
            continue

        ok = step(action, value)
        print("ACTION:", {"action": action, "value": value}, "=>", "OK" if ok else "FAIL")
        render()

    except KeyboardInterrupt:
        print("\nINTERRUPTED")
        break


--- STATE: home ---
Music
Buscador
-----------------------

USER> Ingresa en music y busca diva
ACTION: {'action': 'music', 'value': None} => OK

--- STATE: music_main ---
Biblioteca:
- Aver
- Dnd
- Tú eres mi canción
- No te lloro tanto
-----------------------

USER> Si en la biblioteca no encuentro mi música entre en el buscador y busca diva
ACTION: {'action': 'search_music', 'value': 'diva'} => OK

--- STATE: music_search ---
Buscador música:
Query: diva
Resultados:
-----------------------


INTERRUPTED
